In [ ]:
from cvanmf import denovo
from pathlib import Path
import pandas as pd
import plotnine as pn

In [ ]:
model = denovo.Decomposition.load(
    Path("data/derived/untargetted_metabolomics/metabosignatures/model")
)
model

In [ ]:
sample_md = pd.read_csv(
    "data/source/sample_metadata.csv",
    index_col = "sample_id"
)
sample_md['participant'] = sample_md['participant'].astype("str")
sample_md

In [ ]:
plt_mdl_weight = model.plot_relative_weight(
    group = sample_md['sample_arm'].sort_values(),
    legend_cols_sig=4,
    group_colors=pd.Series(
        dict(
            after_high = "#00BA38",
            after_low = "#28a9e0",
            before_high = "#8BFFAE", 
            before_low = "#aae1f8", 
            midpoint = 'grey'
        )
    ),
    sample_label_size=0,
    heights = [0.2, 0.1, 0.8]
)
plt_mdl_weight.save(
    f"output/figures/supp_metabosigantures.png",
    dpi = 300
)

In [ ]:
plt_weight_arm = (
    model.plot_metadata(
        sample_md['sample_arm']
    )[0] 
    + pn.theme(figure_size=(2, 6))
    + pn.labs(y = "Sample Arm")
    + pn.guides(fill = "none")
    + pn.theme(panel_spacing_y=0.035)
)
plt_weight_arm.save(
    "output/figures/supp_metabosignatures_weight.png",
    dpi = 300
)
plt_weight_arm

## Look at correlations to taxa / genera

In [ ]:
tax_lvl = {
    name: pd.read_csv(
        f'data/source/microbiome/taxa/MGS.matL{lvl}.txt',
        sep = "\t",
        index_col = 0
    )
    .drop(labels = ["-1"], errors = "ignore")
    .rename(columns=lambda x: x.upper())
    for name, lvl in [
        ("mgs", 7),
        ("species", 6),
        ("genus", 5)
    ]
}


In [ ]:
# Correlate each level to signatures
# import pingouin as pg
# import numpy as np

# lvl = "mgs"
# prev_filter = 60

# tax_scale = tax_lvl[lvl] / tax_lvl[lvl].sum()
# tax_scale = tax_scale.drop(columns=["PLATE-1-NC"])
# assert np.allclose(tax_scale.sum(), 1.0)
# # Filter to only quite common speies
# tax_scale = tax_scale.loc[((tax_scale > 0).sum(axis=1) > prev_filter), :]

# df = (
#     tax_scale
#     .T
#     .replace(0, np.nan)
#     .merge(
#         model.scaled("h").T,
#         right_index=True,
#         left_index=True,
#         how="left"
#     )
# )
# res = pg.pairwise_corr(
#     df,
#     columns=[list(tax_scale.index), None],
#     padjust = "fdr_bh",
#     method = "spearman"
# )

In [ ]:
# More conservative correction - pingouins seems to apply within a group
# than to all comparisons
# from pingouin.multicomp import fdr
# res['p_corr2'] = fdr(res['p_unc'])[1]
# res.sort_values(by="p_unc", ascending = True)

In [ ]:
# As a function to apply to MGS / Species / Genus
import pingouin as pg
import numpy as np

def correlate_tax_metabosig(
        model,
        tax,
        name = "taxon",
        prevalence = 60
):
    """Simple correlation of taxa and metabosignatures, using spearman
    correlation and FDR BH correction"""

    # Correlate each level to signatures

    tax_scale = tax / tax.sum()
    tax_scale = tax_scale.drop(columns=["PLATE-1-NC"])
    assert np.allclose(tax_scale.sum(), 1.0)
    # Filter to only quite common speies
    tax_scale = tax_scale.loc[
        ((tax_scale > 0).sum(axis=1) > prevalence),
        :
    ]

    df = (
        tax_scale
        .T
        .replace(0, np.nan)
        .merge(
                model.scaled("h").T,
                right_index=True,
                left_index=True,
                how="left"
        )
    )
    res = pg.pairwise_corr(
        df,
        columns=[list(tax_scale.index), None],
        padjust = "fdr_bh",
        method = "spearman"
    )

    from pingouin.multicomp import fdr
    res['p_corr2'] = fdr(res['p_unc'])[1]
    res.sort_values(by="p_unc", ascending = True)
    return res.assign(tax_rank = name)


all_res = pd.concat(
    [correlate_tax_metabosig(model, x, name, 60)
     for name, x
     in tax_lvl.items()
    ]
)


In [ ]:
# Summarise number of correlations at each rank
sig_count = (
    all_res
    .groupby(by=["tax_rank", "Y"])
    .apply(lambda x: pd.Series(dict(
        significant=(x['p_corr2'] < 0.05).sum(),
        total=x.shape[0],
        mean_r=x[x['p_corr2'] < 0.05]['r'].mean(),
        r_range=x['r']
    )))
)
sig_count['prop'] = sig_count['significant'] / sig_count['total']
sig_count = sig_count.reset_index()
sig_count

In [ ]:
r_abs = sig_count['mean_r'].abs().max()

(
    pn.ggplot(
        sig_count,
        pn.aes(
            x = "Y",
            y = "tax_rank"
        )
    ) +
    pn.geom_tile(
        pn.aes(
            fill = "mean_r"
        )
    ) +
    pn.geom_text(
        pn.aes(
            label = "significant"
        )
    ) +
    pn.ggtitle("Correlations with Taxa") +
    pn.labs(
        y = "Rank",
        x = "Fecal Metabolite Signature"
    ) +
    pn.guides(
        fill = pn.guide_colorbar(title = "Mean r")
    ) +
    pn.scale_fill_distiller(
        palette = "RdBu",
        type = "diverging",
        limits = (-r_abs, r_abs)
    ) +
    pn.theme(
        figure_size=(4, 2)
    )
)

In [ ]:
(
    pn.ggplot(
        all_res[all_res['p_corr2'] < 0.05]
    ) +
    pn.geom_boxplot(
        pn.aes(
            x = "Y",
            y = "r",
            fill = "Y"
        ),
        varwidth = True
    ) +
    pn.facet_wrap(
        facets = "tax_rank"
    ) +
    pn.labs(
        title = "r for significant correlations",
        x = "Signature",
        y = "Spearman's r"
    ) +
    pn.theme(
        figure_size=(4, 2.5),
        axis_text_x=pn.element_blank(),
        axis_ticks_x=pn.element_blank()
    )
)

In [ ]:
(
    pn.ggplot(
        all_res
    ) +
    pn.geom_point(
        pn.aes(
            y = "-np.log(p_unc)",
            x = "r",
            color = "p_corr2 < 0.05",
            size = "p_corr2 < 0.05"
        )
    ) +
    pn.facet_grid(
        rows = "tax_rank",
        cols = "Y"
    ) +
    pn.scale_color_manual(
        values = ["#f04f4f", "#8d8d8d"],
        limits = [True, False]
    ) +
    pn.scale_size_discrete(
        values = [0.1, 1],
        limits = [False, True],
        range = (0, 1)
    ) +
    pn.guides(
        size = "none",
        color = pn.guide_legend(title = "adjusted p < 0.05")
    ) +
    pn.labs(
        title = "Spearman Correlations",
        y = "-log(p)",
        x = "r"
    ) +
    pn.theme_538() +
    pn.theme(
        figure_size=(8, 4),
        legend_position="bottom"
    )
)

In [ ]:
# The above plotting as a function, to apply at MGS and Genus
def plot_correlations(
        all_res
):
    """Plot correlations between taxa and metabolites"""
    sig_count = (
        all_res
        .groupby(by=["tax_rank", "Y"])
        .apply(lambda x: pd.Series(dict(
            significant=(x['p_corr2'] < 0.05).sum(),
            total=x.shape[0],
            mean_r=x[x['p_corr2'] < 0.05]['r'].mean(),
            r_range=x['r']
        )))
    )
    sig_count['prop'] = sig_count['significant'] / sig_count['total']
    sig_count = sig_count.reset_index()
    
    r_abs = sig_count['mean_r'].abs().max()

    plt_hmap = (
        pn.ggplot(
            sig_count,
            pn.aes(
                x = "Y",
                y = "tax_rank"
            )
        ) +
        pn.geom_tile(
            pn.aes(
                fill = "mean_r"
            )
        ) +
        pn.geom_text(
            pn.aes(
                label = "significant"
            )
        ) +
        pn.ggtitle("Correlations with Taxa") +
        pn.labs(
            y = "Rank",
            x = "Fecal Metabolite Signature"
        ) +
        pn.guides(
            fill = pn.guide_colorbar(title = "Mean r")
        ) +
        pn.scale_fill_distiller(
            palette = "RdBu",
            type = "diverging",
            limits = (-r_abs, r_abs)
        ) +
        pn.theme(
            figure_size=(4, 2)
        )
    )

    plt_volcano = (
        pn.ggplot(
            all_res
        ) +
        pn.geom_point(
            pn.aes(
                y = "-np.log(p_unc)",
                x = "r",
                color = "p_corr2 < 0.05",
                size = "p_corr2 < 0.05"
            )
        ) +
        pn.facet_grid(
            rows = "tax_rank",
            cols = "Y"
        ) +
        pn.scale_color_manual(
            values = ["#f04f4f", "#8d8d8d"],
            limits = [True, False]
        ) +
        pn.scale_size_discrete(
            values = [0.1, 1],
            limits = [False, True],
            range = (0, 1)
        ) +
        pn.guides(
            size = "none",
            color = pn.guide_legend(title = "adjusted p < 0.05")
        ) +
        pn.labs(
            title = "Spearman Correlations",
            y = "-log(p)",
            x = "r"
        ) +
        pn.theme_538() +
        pn.theme(
            figure_size=(8, 4),
            legend_position="bottom"
        )
    )

    return dict(
        heatmap = plt_hmap,
        volcano = plt_volcano,
        count_data = sig_count
    )

correlations = {
    name: correlate_tax_metabosig(
        model,
        tax = x,
        name = name
    ) for name, x in tax_lvl.items()
}

correlations_plots = {
    name: plot_correlations(x)
    for name, x in correlations.items()
}

In [ ]:
plt_species_corr = (
    correlations_plots['species']['volcano']
    + pn.theme(
        figure_size = (7.5, 2.5),
        plot_background=pn.element_blank()
    )
    + pn.scale_x_continuous(
        breaks = [-0.5, 0, 0.5]
    )
)
plt_species_corr.save(
    "output/figures/supp_metabosignatures_corr.png",
    dpi = 300
)
plt_species_corr

In [ ]:
# Top 10 strongest correlations at species level
top10_spec = (
    all_res.loc[
        (all_res['Y'] == "S6") &
        (all_res['p_corr2'] < 0.05) &
        (all_res['tax_rank'] == "species"),
        :
    ]
    .sort_values(by='r', ascending = False)
).iloc[0:10]["X"].tolist()

l_spec = [
    x.split(';')[-1] for x in top10_spec
    if "Clostridia" in x
]

", ".join(l_spec)

In [ ]:
top10_spec

In [ ]:
spec_df = (
    all_res.loc[
        (all_res['Y'] == "S6") &
        (all_res['p_corr2'] < 0.05) &
        (all_res['tax_rank'] == "species"),
        :
    ]
    .sort_values(by='r', ascending = False)
).iloc[0:10]
spec_df

## Gene enrichment?

The S6 associated taxa are typical butyrate producers.
Are some key genes for butyrate production enriched in S6 associated MGS?

Use Fisher exact test to test for difference in presence / absence of genes
in taxa significantly correlated / not, to try to estimate if there's any
functional driver for this signature.

In [ ]:
# Apply to MGS averages - assume any >0 means present - a bit of a simplifying
# assumption

cazys = pd.read_csv(
    "data/derived/function/function_counts/CAZY.tsv",
    sep = "\t",
    index_col = 0
).fillna(0) > 0
cazys

In [ ]:
# This has lost the MSG identities in column headings, relabel
cazys.columns = [f'MGS.{x}' for x in cazys.columns]

In [ ]:
# Above as a funtion to test for enrichment
# This should be intergrated into yoke also
from scipy.stats import fisher_exact, barnard_exact
from pingouin.multicomp import fdr

def barnard_tuple(beres):
    return (beres.statistic, beres.pvalue)

def fisher_enrichment(
        mgs_func_table,
        association_table,
        signature,
        assoc_alpha = 0.05
):
    long_fn = (
        mgs_func_table
        .stack()
        .to_frame("present")
        .reset_index(names=['gene', 'mgs'])
    )

    association_table['assoc'] = association_table['p_corr2'] < 0.05
    mgs_s6 = association_table.loc[
        (association_table['tax_rank'] == "mgs") &
        (association_table['Y'] == signature), :
    ]
    # Select down to only MGS
    mgs_s6['mgs'] = mgs_s6['X'].apply(lambda x: x.split(";")[-1])
    mgs_s6 = mgs_s6.loc[mgs_s6['mgs'] != "?", :]

    long_fn_md = long_fn.merge(
        mgs_s6[['mgs', 'p_corr2', 'r', 'assoc']],
        left_on="mgs", 
        right_on="mgs", 
        how="left"
    )
    long_fn_md = long_fn_md.dropna(
        axis = 0,
        subset=['assoc']
    )
    long_fn_md['assoc'] = pd.Categorical(
        long_fn_md['assoc'],
        categories=[True, False]
    )
    long_fn_md['present'] = pd.Categorical(
        long_fn_md['present'],
        categories=[True, False]
    )

    fish_res = (
        long_fn_md
        .groupby(by=["gene"])
        .apply(lambda x: pd.Series(fisher_exact(
            pd.crosstab(x['assoc'], x['present'], dropna=False))
        ))
        .rename(columns={0: "fisher_stat", 1: "fisher_p"})
    )
    fish_res['fisher_p_adj'] = fdr(
        fish_res['fisher_p']
    )[1]

    barn_res = (
        long_fn_md
        .groupby(by=["gene"])
        .apply(lambda x: pd.Series(barnard_tuple(barnard_exact(
            pd.crosstab(x['assoc'], x['present'], dropna=False))
        )))
        .rename(columns={0: "barnard_stat", 1: "barnard_p"})
    )
    barn_res['barnard_p_adj'] = fdr(
        barn_res['barnard_p']
    )[1]

    return pd.concat([fish_res, barn_res], axis = 1)

all_fish = pd.concat([
    fisher_enrichment(
        cazys,
        association_table=all_res,
        signature=x
    ).assign(signature=x)
    for x in
    model.names
])


In [ ]:
all_fish.sort_values(by="barnard_p_adj", ascending=True)

In [ ]:
# KEGG
ko = pd.read_csv(
    "data/derived/function/function_counts/KEGG.tsv",
    sep = "\t",
    index_col=0
).fillna(0) > 0
ko.columns = [f'MGS.{x}' for x in ko.columns]

In [ ]:
butyrate_genes = ["K19709", "K00929"]
ko_butyrate = ko.loc[butyrate_genes, :]

In [ ]:
ko_fish = pd.concat([
    fisher_enrichment(
        ko_butyrate,
        association_table=all_res,
        signature=x
    ).assign(signature=x)
    for x in
    model.names
])

In [ ]:
ko_fish.sort_values(by="barnard_p_adj", ascending=True)

No observable enrichments for butyrate genes in S6 associated MGS